<a href="https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
from google.colab import userdata
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
path = f"{rel}/fact_content_daily_performance/month={MONTH}/data_0.parquet"

date_col, client_col, content_col = "report_date", "client_hash_id", "content_hash_id"
clicks_col, impr_col, avgpos_col, avail_col = "gsc_clicks", "gsc_impressions", "gsc_avg_position", "gsc_data_available"

In [4]:
dim_content_path = f"{rel}/dim_content.parquet"
dc_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{dim_content_path}') LIMIT 0").df()
print(dc_schema)

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: flag content whose search visibility isn't converting to clicks at the
rate its position should predict, or that has gone stale since its last
content update. Two signals back this, checked against real data first:

- Signal 1 (flag-linked — staleness, behind the refresh flags): days since
  `content_updated_date`.
- Signal 2 (flag-linked — CTR-vs-position, behind the CTR-fix logic): CTR
  relative to position bucket.

Reason codes the rule can output:
- CTR_UNDERPERFORM_FOR_POSITION — good position (top 10), CTR below the
  bucket's average
- STALE_CONTENT — no content update in 180+ days
- HIGH_VOLUME_LOW_CONVERT — high impressions overall, neither of the above triggers
- LOW_PRIORITY — none of the above

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

staleness = con.sql(f"""
    SELECT {content_col}, content_updated_date, is_published, is_deleted,
           DATE_DIFF('day', content_updated_date, DATE '2026-03-31') AS staleness_days
    FROM read_parquet('{dim_content_path}')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

perf_ctr = con.sql(f"""
    SELECT {content_col} AS cid, AVG({clicks_col} / NULLIF({impr_col}, 0)) AS avg_ctr
    FROM read_parquet('{path}')
    WHERE {avail_col} IS TRUE
    GROUP BY 1
""").df()

merged = staleness.merge(perf_ctr, left_on=content_col, right_on='cid', how='inner')
merged['bucket'] = pd.cut(merged['staleness_days'], bins=[-1,30,90,180,99999],
                           labels=['0-30d','31-90d','91-180d','180d+'])
bucket_table = merged.groupby('bucket', observed=True).agg(n=('avg_ctr','size'), mean_ctr=('avg_ctr','mean')).reset_index()
print(bucket_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    bucket      n  mean_ctr
0    0-30d    672  0.004316
1   31-90d  25577  0.002424
2  91-180d   1324  0.020790
3    180d+    228  0.002444


In [6]:
ctr_pos = con.sql(f"""
    SELECT
        CASE WHEN {avgpos_col} <= 3 THEN '1-3'
             WHEN {avgpos_col} <= 10 THEN '4-10'
             WHEN {avgpos_col} <= 20 THEN '11-20'
             ELSE '20+' END AS position_bucket,
        AVG({clicks_col} / NULLIF({impr_col}, 0)) AS avg_ctr,
        COUNT(*) AS n
    FROM read_parquet('{path}')
    WHERE {avail_col} IS TRUE
    GROUP BY 1 ORDER BY 1
""").df()
print(ctr_pos)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket   avg_ctr        n
0             1-3  0.004756   727362
1           11-20  0.002770   519223
2             20+  0.001289   908354
3            4-10  0.003473  1456122


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import os

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

agg = con.sql(f"""
    SELECT {content_col} AS content_id, {client_col} AS client_id,
           AVG({clicks_col}) AS avg_clicks,
           AVG({impr_col}) AS avg_impressions,
           AVG({avgpos_col}) AS avg_position,
           AVG({clicks_col} / NULLIF({impr_col}, 0)) AS avg_ctr
    FROM read_parquet('{path}')
    WHERE {avail_col} IS TRUE
    GROUP BY 1,2
""").df()

content_meta = con.sql(f"""
    SELECT {content_col} AS content_id, content_updated_date, search_volume,
           is_published, is_deleted
    FROM read_parquet('{dim_content_path}')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

rule_df = agg.merge(content_meta, on='content_id', how='inner')
rule_df['staleness_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(rule_df['content_updated_date'])).dt.days

rule_df['score'] = (rule_df['avg_impressions'] * (1 / rule_df['avg_position'].clip(lower=1))) * (1 - rule_df['avg_ctr'].fillna(0))

median_impr = rule_df['avg_impressions'].median()

def reason_code(row):
    if row['avg_position'] <= 10 and (row['avg_ctr'] or 0) < 0.02:
        return "CTR_UNDERPERFORM_FOR_POSITION"
    elif row['staleness_days'] > 180:
        return "STALE_CONTENT"
    elif row['avg_impressions'] > median_impr:
        return "HIGH_VOLUME_LOW_CONVERT"
    else:
        return "LOW_PRIORITY"

rule_df['reason_code'] = rule_df.apply(reason_code, axis=1)
rule_df['action'] = rule_df['reason_code'].map({
    "CTR_UNDERPERFORM_FOR_POSITION": "FIX_CTR",
    "STALE_CONTENT": "REFRESH_CONTENT",
    "HIGH_VOLUME_LOW_CONVERT": "REVIEW_CONTENT",
    "LOW_PRIORITY": "NO_ACTION"
})

queue = rule_df.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Rows written:", len(queue))
queue.head(20)

Rows written: 176568


,content_id,client_id,avg_clicks,avg_impressions,avg_position,avg_ctr,content_updated_date,search_volume,is_published,is_deleted,staleness_days,score,reason_code,action
0,content_eadb33b5df496f4a,client_e547b89c05043229,195.448276,21280.137931,2.383011,0.010771,2026-06-12,390,True,False,-73,8833.750427,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
1,content_ec2e0346994fb5a5,client_e547b89c05043229,51.034483,8457.793103,2.854514,0.006100,2026-06-12,0,True,False,-73,2944.879932,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
2,content_0e03de7680314cd5,client_e547b89c05043229,24.827586,7631.379310,2.675217,0.005032,2026-06-12,110,True,False,-73,2838.265683,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
3,content_4ffe18112a5642e3,client_e547b89c05043229,20.206897,6447.689655,2.331060,0.004094,2026-06-12,40,True,False,-73,2754.666385,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
4,content_8d7d99f109e19aa2,client_e547b89c05043229,9.965517,7017.137931,2.563756,0.003598,2026-06-12,70,True,False,-73,2727.205596,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
5,content_b13e95d379c78818,client_62f4a7e64f5e0096,4.870968,2455.516129,1.139193,0.002564,2026-07-04,10,True,False,-95,2149.959707,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
6,content_f107e54b10b43725,client_62f4a7e64f5e0096,32.129032,6322.483871,3.186054,0.005700,2026-07-03,0,True,False,-94,1973.113361,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
7,content_7172a7fad43f0998,client_62f4a7e64f5e0096,27.806452,6640.870968,3.367835,0.004412,2026-07-03,10,True,False,-94,1963.151407,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
8,content_306bc78dff1eb683,client_e547b89c05043229,1.206897,2786.931034,1.488604,0.000431,2026-06-22,40500,True,False,-83,1871.370397,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR
9,content_987d251ee617d9c6,client_73cda7b4e4f265ea,30.322581,4929.225806,2.823429,0.006673,2026-07-03,2900,True,False,-94,1734.180138,CTR_UNDERPERFORM_FOR_POSITION,FIX_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_eadb33b5df496f4a — action FIX_CTR — reason CTR_UNDERPERFORM_FOR_POSITION — confidence: high (21,280 avg impressions, position ~2.4) — wrong if: CTR this low at top position reflects a snippet/featured-answer cannibalizing clicks, not a fixable title/meta issue.
2. content_ec2e0346994fb5a5 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (8,458 impressions) — wrong if: query intent is navigational and users don't need to click through.
3. content_0e03de7680314cd5 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (7,631 impressions) — wrong if: the result already answers the query in the SERP snippet.
4. content_4ffe18112a5642e3 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (6,448 impressions) — wrong if: low search_volume (40) means this is a one-off spike, not a stable pattern.
5. content_8d7d99f109e19aa2 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (7,017 impressions) — wrong if: clicks are being captured by an AI-overview/assistant channel instead of organic (check sessions_ai for this content).
6. content_b13e95d379c78818 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (2,456 impressions, lower volume) — wrong if: search_volume (10) is too small for the CTR estimate to be stable.
7. content_f107e54b10b43725 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (6,322 impressions) — wrong if: position 3.2 already reflects a competitive SERP with heavy ad/AI presence suppressing all organic CTR.
8. content_7172a7fad43f0998 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (6,641 impressions) — wrong if: same SERP-crowding effect as row 7 (same client).
9. content_306bc78dff1eb683 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (2,787 impressions, but very low avg_clicks=1.2) — wrong if: search_volume (40,500) suggests this keyword is highly competitive/ambiguous intent, not a title/meta problem.
10. content_987d251ee617d9c6 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (4,929 impressions) — wrong if: this is a branded query where users already know the destination and skip the click.
11. content_acbcc847f8996314 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (5,510 impressions, low clicks=8.5) — wrong if: SERP has heavy People-Also-Ask boxes absorbing attention at this position.
12. content_512dbad65bd5ade9 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (4,979 impressions) — wrong if: CTR here (0.017) is actually close to bucket-normal and the flag is a marginal call.
13. content_545bb6cc7081ded3 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (4,238 impressions) — wrong if: content_created recently relative to ranking, so CTR simply hasn't stabilized yet.
14. content_c46df0fa61530d86 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (2,428 impressions, very low clicks=1.4) — wrong if: search_volume (12,100) signals a broad/ambiguous keyword where low CTR is structural, not fixable.
15. content_b2b85c287474668d — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (2,252 impressions) — wrong if: sample size this small makes the CTR estimate unstable.
16. content_e7b5dd4dff461ad2 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (6,614 impressions) — wrong if: position 4.5 already sits below the fold on mobile SERPs, capping CTR regardless of title/meta.
17. content_fd2117c2c6790e4b — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (4,876 impressions) — wrong if: this keyword cluster overlaps with a stronger-ranking page from the same client cannibalizing clicks.
18. content_34a70fea29d15f24 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: low (4,614 impressions but only 1.4 avg_clicks — very thin) — wrong if: this small a click count makes the CTR estimate essentially noise.
19. content_85703b835ab9e744 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: high (3,899 impressions) — wrong if: CTR (0.0094) is actually near the norm for position ~2.7 in a more competitive niche.
20. content_b99ea6861864dea5 — FIX_CTR — CTR_UNDERPERFORM_FOR_POSITION — confidence: medium (6,269 impressions, low clicks=11.6) — wrong if: position 4.45 is borderline and small position changes explain the gap, not a content problem.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check finding: staleness_days came back negative for every row in the
queue (content_updated_date postdates the March 2026 decision month). This is
because dim_content is a single frozen snapshot, not a monthly history, so it
reflects updates that hadn't happened yet as of the decision point. This
signal is NOT decision-time safe as implemented, and STALE_CONTENT never
actually fired in this run because CTR_UNDERPERFORM_FOR_POSITION always
matched first. Fix: drop staleness-based reasoning entirely for past-month
analysis unless a historical snapshot of dim_content per month becomes
available — do not trust content_updated_date as "known" for any month before
the export date.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Inputs used in scoring:", ['avg_clicks','avg_impressions','avg_position','avg_ctr','staleness_days','search_volume'])
print("All derived from month=2026-03 GSC aggregates (filtered gsc_data_available IS TRUE) or a static content attribute (content_updated_date). No future month, no product flag, no label-derived column used.")

print("Fact table path used:", path)


Inputs used in scoring: ['avg_clicks', 'avg_impressions', 'avg_position', 'avg_ctr', 'staleness_days', 'search_volume']
All derived from month=2026-03 GSC aggregates (filtered gsc_data_available IS TRUE) or a static content attribute (content_updated_date). No future month, no product flag, no label-derived column used.
Fact table path used: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.